# Map Builder


<b> Objective: </b>
Create a base map of Greenland

----------------------

<i><b>Author: </b> Salvador Palma <br>
<b>Institution: </b> Copenhagen University </i>


### Building the Basemap

Creates `Data/GreenlandMapPlain.png` from [Natural Earth](https://www.naturalearthdata.com/) vector data.

**Source archives**

[.zip](https://naciscdn.org/naturalearth/10m/physical/ne_10m_land.zip) : `Data/naturalearth/ne_land/ne_10m_land.shp`<br>
[.zip](https://naciscdn.org/naturalearth/10m/physical/ne_10m_glaciated_areas.zip) : `Data/naturalearth/ne_glacier/ne_10m_glaciated_areas.shp`


In [1]:
import struct
from pathlib import Path

import numpy as np
from matplotlib.figure import Figure
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath

SHAPEFILES = {
    "land":    Path("Data/naturalearth/ne_land/ne_10m_land.shp"),
    "glacier": Path("Data/naturalearth/ne_glacier/ne_10m_glaciated_areas.shp"),
}


def ReadShapefilePolygons(shpPath: Path) -> list[list[list[tuple[float, float]]]]:
    shpBytes = Path(shpPath).read_bytes()

    records = []
    pos, n = 100, len(shpBytes)

    while pos < n:
        _, contentLen = struct.unpack_from(">ii", shpBytes, pos)
        pos += 8
        end = pos + contentLen * 2

        shapeType = struct.unpack_from("<i", shpBytes, pos)[0]
        if shapeType not in (3, 5):
            pos = end
            continue

        numParts, numPoints = struct.unpack_from("<ii", shpBytes, pos + 36)
        partsOff = pos + 44
        ptsOff = partsOff + numParts * 4

        partIdx = list(struct.unpack_from(f"<{numParts}i", shpBytes, partsOff))
        partIdx.append(numPoints)
        coords = struct.unpack_from(f"<{numPoints * 2}d", shpBytes, ptsOff)

        rings = []
        for i in range(numParts):
            a, b = partIdx[i], partIdx[i + 1]
            ring = [(coords[j * 2], coords[j * 2 + 1]) for j in range(a, b)]
            if len(ring) >= 3:
                rings.append(ring)
        if rings:
            records.append(rings)

        pos = end

    return records

In [6]:
#Rendering limits
BASEMAP_BBOX = (-400_000.0, 6_500_000.0, 1_200_000.0, 9_400_000.0)
LONLAT_WINDOW = (-75.0, 58.5, -10.0, 84.0)

#Colors
MAP_SEA = "#cfe0f0"
MAP_LAND = "#c8c0ac"
MAP_ICE = "#fafbfc"
MAP_COAST = "#6c7a8a"
MAP_ICE_EDGE = "none"

#Resolution
MAP_SIZE = (2000, 3625)
MAP_DPI = 100


def lonlatToUTM24N(lon, lat) -> tuple[np.ndarray, np.ndarray]:
    lon, lat = np.asarray(lon, dtype=float), np.asarray(lat, dtype=float)
    lon0, k0, a, e2 = -39.0, 0.9996, 6378137.0, 0.00669438

    lat_r = np.radians(lat)
    dlon = np.radians(((lon - lon0 + 180.0) % 360.0) - 180.0)
    sin_lat, cos_lat = np.sin(lat_r), np.cos(lat_r)
    n = a / np.sqrt(1 - e2 * sin_lat ** 2)
    t = np.tan(lat_r) ** 2
    c = e2 / (1 - e2) * cos_lat ** 2
    aa = dlon * cos_lat
    m = a * ((1 - e2 / 4 - 3 * e2 ** 2 / 64) * lat_r - (3 * e2 / 8 + 3 * e2 ** 2 / 32) * np.sin(2 * lat_r) + (15 * e2 ** 2 / 256) * np.sin(4 * lat_r))
    x = k0 * n * (aa + (1 - t + c) * aa ** 3 / 6) + 500000
    y = k0 * (m + n * sin_lat / cos_lat * (aa ** 2 / 2 + (5 - t + 9 * c) * aa ** 4 / 24))
    return x, y


def RingsInWindow(file: str) -> list[np.ndarray]:
    wMinLon, wMinLat, wMaxLon, wMaxLat = LONLAT_WINDOW

    rings = []
    for record in ReadShapefilePolygons(SHAPEFILES[file]):
        for ring in record:
            lon = np.fromiter((p[0] for p in ring), float, len(ring))
            lat = np.fromiter((p[1] for p in ring), float, len(ring))
            if lon.min() < wMinLon or lon.max() > wMaxLon:
                continue
            if lat.min() < wMinLat or lat.max() > wMaxLat:
                continue
            x, y = lonlatToUTM24N(lon, lat)
            rings.append(np.column_stack([x, y]))
    return rings


def AddRings(ax, rings, facecolor, edgecolor, lw, zorder, clipPath=None):
    verts, codes = [], []
    for r in rings:
        verts.append(r)
        codes.append(np.full(len(r), MplPath.LINETO))
        codes[-1][0] = MplPath.MOVETO
    if not verts:
        return None

    patch = PathPatch(MplPath(np.vstack(verts), np.concatenate(codes)),facecolor=facecolor, edgecolor=edgecolor, linewidth=lw,zorder=zorder, joinstyle="round", capstyle="round")
    ax.add_patch(patch)
    if clipPath is not None:
        patch.set_clip_path(clipPath)
    return patch


def BuildBasemap(outPath: Path = Path("Data/GreenlandMapPlain.png")) -> Path:

    land = RingsInWindow("land")
    ice = RingsInWindow("glacier")
    
    print(f"Land Rings = {len(land)}")
    print(f"Glacier Rings = {len(ice)}")
    
    pts = np.vstack(land)
    spanY = pts[:, 1].max() - pts[:, 1].min()
    spanX = pts[:, 0].max() - pts[:, 0].min()
    print(f"Land Extent: {spanX/1e3:.0f} x {spanY/1e3:.0f} km")
   

    minx, miny, maxx, maxy = BASEMAP_BBOX
    fig = Figure(figsize=(MAP_SIZE[0] / MAP_DPI, MAP_SIZE[1] / MAP_DPI), dpi=MAP_DPI)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_facecolor(MAP_SEA)
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_aspect("equal")
    ax.set_axis_off()

    
    landPatch = AddRings(ax, land, MAP_LAND, MAP_COAST, 0.6, 2)
    AddRings(ax, ice, MAP_ICE, MAP_ICE_EDGE, 0.4, 3, clipPath=landPatch)

    outPath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outPath, dpi=MAP_DPI, facecolor=MAP_SEA, pad_inches=0)
    return outPath

In [7]:
BASEMAP_PATH = Path("Data/GreenlandMapPlain.png")
BuildBasemap(BASEMAP_PATH)

Land Rings = 213
Glacier Rings = 745
Land Extent: 2928 x 2653 km


WindowsPath('Data/GreenlandMapPlain.png')